## Wrapper Based Feature Selection
- Unlike filter based methods, wrapper based methods use estimator class rather than a scoring function.

* 1. Recursive Feature Estimation(RFE)
    * Uses an estimator to recursively remove features.
        * Initially fits an estimator on all features.
    * Obtains feature importance from the estimator and removes the least important feature.
    * Repeats the process by removing features one by one, until desired number of features are obtained.

    * Use RFECV if we do not want to specify the desired number of features in RFE .
    * It performs RFE in a cross-validation loop to find the optimal number of features.

# 🔪 Topic 1: Recursive Feature Elimination (RFE)

**Core Concept (Basic Level):** 
RFE ek Reality TV Show (jaise Bigg Boss ya Roadies) ki tarah kaam karta hai. 
1. Shuru me saare features (contestants) model ke andar aate hain. 
2. Model sabko test karta hai aur sabko ek "Importance Score" deta hai.
3. Jiska score sabse kam hota hai (weakest contestant), usko **eliminate** (bahar) kar diya jata hai.
4. Bacha hua game wapas se start hota hai, naye scores milte hain, aur fir se sabse weak feature eliminate hota hai. 
5. Ye tab tak chalta hai jab tak humari desired number of features (jo humne set ki hai) na bach jayein.

### 🔴 High-Level Mathematics (Under the Hood):
Aap sochoge ki *"Bhai ek hi baar me sabse weak features ko kyun nahi nikal dete? Ye baar-baar train aur drop (Recursive) kyun karna?"*
Yahi iski sabse badi Maths hai: **Collinearity and Weight Shifting**.

Maan lijiye hum Linear Regression use kar rahe hain, jiska equation hai:
$$y = w_1x_1 + w_2x_2 + \dots + w_nx_n$$
Yahan $w$ (weights) features ki importance batate hain. 
Agar $x_1$ aur $x_2$ aapas me bohot closely related hain (Highly Correlated), toh model dono me weights aadha-aadha baant dega. Agar hum dono ko ek sath drop kar denge, toh data ka loss hoga. 
Lekin RFE me, agar hum pehle sirf $x_2$ ko drop karte hain, toh agle round (retraining) me $x_1$ ka weight achanak badh jayega! Isliye har elimination ke baad model ko **re-train** karna zaroori hai taaki bache hue features ke weights dobara adjust (re-calculate) ho sakein.

**Feature Importance kahan se aati hai?**
* **Linear Models (SVM, Logistic Regression):** Inme coefficients (`coef_`) hote hain. Jiske coefficient ka absolute size $|w_i|$ bada hota hai, wo feature important hota hai.
* **Tree Models (Random Forest, XGBoost):** Inme `feature_importances_` attribute hota hai jo calculate karta hai ki kis feature ne data ki impurity (Gini Impurity ya Entropy) ko sabse zyada kam kiya.

---

In [2]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.feature_selection import RFE
from sklearn.tree import DecisionTreeClassifier

def rfe_implementation():
    # 1. Dataset banaya jisme 10 features hain, par asli kaam ke sirf 4 hain
    X, y = make_classification(n_samples=1000, n_features=10, n_informative=4, random_state=42)
    feature_names = [f"Sensor_{i}" for i in range(1, 11)]
    X_df = pd.DataFrame(X, columns=feature_names)
    
    # 2. Estimator define kiya (Tree model kyunki ye non-linear data samajhta hai)
    estimator = DecisionTreeClassifier(random_state=42)
    
    # 3. RFE Initialize kiya (Hum model ko bol rahe hain: Saare hatao, bas best 4 rakho)
    # step=1 ka matlab hai ek baar me sirf 1 feature ko drop karna (Highest accuracy, par thoda slow)
    rfe_selector = RFE(estimator=estimator, n_features_to_select=4, step=1)
    
    # 4. Fit and Transform
    X_selected_matrix = rfe_selector.fit_transform(X_df, y)
    
    # 5. Extract Feature Names and Rankings
    selected_features = rfe_selector.get_feature_names_out(X_df.columns)
    X_final = pd.DataFrame(X_selected_matrix, columns=selected_features)
    
    # 6. Rank Sheet Banana (Bohot important industry practice)
    # Rank 1 ka matlab wo select ho gaya. Rank 2, 3.. ka matlab wo kitni jaldi eliminate hua.
    rank_sheet = pd.DataFrame({
        'Feature': X_df.columns,
        'Selected': rfe_selector.support_,
        'Rank': rfe_selector.ranking_
    }).sort_values(by='Rank')
    
    print("--- RFE Feature Ranking ---")
    print(rank_sheet)
    print(f"\nFinal Selected Dataset Shape: {X_final.shape}")

rfe_implementation()

--- RFE Feature Ranking ---
     Feature  Selected  Rank
0   Sensor_1      True     1
2   Sensor_3      True     1
5   Sensor_6      True     1
9  Sensor_10      True     1
4   Sensor_5     False     2
6   Sensor_7     False     3
3   Sensor_4     False     4
7   Sensor_8     False     5
8   Sensor_9     False     6
1   Sensor_2     False     7

Final Selected Dataset Shape: (1000, 4)


# 👑 Topic 2: RFE with Cross-Validation (RFECV)

**Core Concept (Basic Level):** 
Normal RFE me ek problem hai: Humko manually batana padta hai ki `n_features_to_select=4` ya `10`. Par real world me hume thodi pata hai ki data me kitne features actually important hain! Agar humne galat number de diya, toh model kharaab ho jayega.
**RFECV (Recursive Feature Elimination with Cross-Validation)** iska Smart Auto-Pilot version hai. 
Ye model se kehta hai: *"Tu feature drop karta jaa, aur har baar khud ka Test (Cross-Validation) leta jaa. Jis number of features par teri Accuracy/Score sabse highest ho, wahan ruk jana!"*

### 🔴 High-Level Mathematics (Under the Hood):
Ye optimization problem ko solve karta hai. 
Maan lijiye humare paas Total $M$ features hain.
1. **$K$-Fold CV:** Dataset ko $K$ tukdon (folds) me baanta jata hai.
2. Ye $M$ features ke sath train karta hai, validation fold par test karta hai, aur average accuracy nikalta hai.
3. Fir ye weakest feature ko udata hai. Ab features bache $M-1$.
4. Ye $M-1$ features ke sath fir se $K$-Fold CV karta hai aur nayi average accuracy nikalta hai.
5. Ye process $M, M-1, M-2, \dots, 1$ tak chalti hai.
6. Aakhiri me ek Curve (Graph) banta hai: **X-axis** = Number of Features, **Y-axis** = CV Score. 
7. Math function: Ye us curve ka **Argmax** (Highest Peak point) dhoondhta hai. Jis feature count par curve sabse uncha hota hai, wahi optimal number hai.

---


In [3]:
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.feature_selection import RFECV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold

def rfecv_implementation():
    # 1. Dataset (15 features, jisme sirf 5 actually important hain)
    X, y = make_classification(n_samples=800, n_features=15, n_informative=5, random_state=1)
    X_df = pd.DataFrame(X, columns=[f"Var_{i}" for i in range(1, 16)])
    
    # 2. Estimator and Cross-Validation Strategy
    estimator = RandomForestClassifier(random_state=42, n_jobs=-1)
    cv_strategy = StratifiedKFold(n_splits=5) # 5-fold cross-validation
    
    # 3. RFECV Initialize karna
    # scoring='accuracy' (Classification ke liye). Regression hota to 'r2' ya 'neg_mean_squared_error' rakhte
    rfecv = RFECV(estimator=estimator, step=1, cv=cv_strategy, scoring='accuracy', n_jobs=-1)
    
    # 4. Model Training and Feature Selection
    print("🤖 RFECV is finding the optimal number of features... Please wait...")
    X_selected = rfecv.fit_transform(X_df, y)
    
    # 5. Extract Results
    optimal_num = rfecv.n_features_
    selected_names = rfecv.get_feature_names_out(X_df.columns)
    
    print("\n--- RFECV Results ---")
    print(f"Total Features Start me: {X_df.shape[1]}")
    print(f"Optimal Number of Features Found: {optimal_num}")
    print(f"Selected Feature Names: {list(selected_names)}")
    
    # Industry Bonus: Maximum accuracy kitni aayi wo number of features pe?
    best_score = rfecv.cv_results_['mean_test_score'].max()
    print(f"Highest Cross-Validation Accuracy: {best_score * 100:.2f}%")

rfecv_implementation()

🤖 RFECV is finding the optimal number of features... Please wait...

--- RFECV Results ---
Total Features Start me: 15
Optimal Number of Features Found: 6
Selected Feature Names: ['Var_2', 'Var_5', 'Var_8', 'Var_9', 'Var_13', 'Var_14']
Highest Cross-Validation Accuracy: 89.62%


# 🚪 Topic 3: SelectFromModel

**Core Concept (Basic Level):** 
Ye RFE ki tarah baar-baar train (recursive) nahi karta. Ye sirf ek baar model ko train karta hai aur ek strict **Cutoff (Threshold)** set kar deta hai. 
Jaise College ka Admission: *"Jis student ke board exam me 80% se upar marks hain, sirf unko aane do."* Yahan farq nahi padta ki kitne bachhe aayenge—chahe 5 aayein ya 50—sirf cutoff clear karna zaroori hai.

Aap is cutoff ko statically (jaise `max_features=5`) de sakte ho, ya dynamically string format me (jaise `'mean'`, `'median'`, ya `'1.25*mean'`) de sakte ho.

### 🔴 High-Level Mathematics (Under the Hood):
Iska mathematical logic us formula (threshold) me chupa hai:
1. **Model Training:** Model ek baar train hota hai aur saare weights/importances nikalta hai. Let the importances be a vector $I = [i_1, i_2, \dots, i_n]$.
2. **Thresholding (Dynamic):** 
   * Agar aapne `'mean'` diya hai, toh math calculation hogi: $\mu = \frac{1}{n}\sum I$. Jo bhi feature importance $\ge \mu$ hogi, wo select hoga.
   * Agar aapne `'1.25*median'` diya hai, toh pehle saare scores ka median niklega, usko $1.25$ se multiply kiya jayega, aur jo us strict cutoff ko cross karega, wahi aage jayega.

**🔥 L1 Regularization (Lasso) Magic:**
SelectFromModel ka sabse deadly combination **L1 Penalty (Lasso Regression/LinearSVC)** ke sath hota hai. L1 Regularization ki math aisi hoti hai ki wo kachra (unimportant) features ke exact weights ko **Zero (0)** kar deta hai! 
Equation me penalty term hoti hai: $\lambda \sum |w_i|$. Ye constraint weights ko diamond-shaped plane pe push karta hai jisse directly axes (zero) par intersection hoti hai. SelectFromModel wahan bas ek simple check lagata hai: Jiska weight 0 nahi hai, usko rakh lo!

---

In [6]:
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LassoCV

def select_from_model():
    # 1. Regression Dataset Load karna
    data = load_diabetes()
    X = pd.DataFrame(data.data, columns=data.feature_names)
    y = data.target
    
    print(f"Original Features: {list(X.columns)}")
    
    # 2. Estimator banana (LassoCV khud se best regularization math lagata hai)
    # L1 Regularization specifically feature selection ke liye best maana jata hai
    lasso_estimator = LassoCV(cv=5, random_state=42)
    
    # 3. SelectFromModel Initialize karna
    # Hum ek dynamic threshold laga rahe hain: Jo features average (mean) importance
    # se kam se kam 25% zyada important hain (1.25 * mean), bas unhi ko lo!
    sfm_selector = SelectFromModel(estimator=lasso_estimator, threshold='1.25*mean')
    
    # 4. Fit and Transform
    X_strict_selected = sfm_selector.fit_transform(X, y)
    
    # 5. Check what survived the cutoff
    surviving_features = sfm_selector.get_feature_names_out(X.columns)
    
    print("\n--- SelectFromModel (Lasso L1) Results ---")
    print(f"Threshold Formula Used: 1.25 * Mean Importance")
    print(f"Number of Features Survived: {X_strict_selected.shape[1]}")
    print(f"Elite Selected Features: {list(surviving_features)}")
    
    # Under the hood logic dekhne ke liye:
    # Jin features ka coefficient cutoff se upar tha, unki value print kar rahe hain
    feature_importances = sfm_selector.estimator_.coef_
    importance_df = pd.DataFrame({
        'Feature': X.columns,
        'Absolute Weight (Importance)': abs(feature_importances)
    })
    
    # Calculate exactly what the threshold value was computationally
    threshold_value = sfm_selector.threshold_
    print(f"\nCalculated Cutoff Score was: {threshold_value:.4f}")
    print("\nImportance Breakdown:")
    print(importance_df.sort_values(by='Absolute Weight (Importance)', ascending=False))

select_from_model()

Original Features: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']

--- SelectFromModel (Lasso L1) Results ---
Threshold Formula Used: 1.25 * Mean Importance
Number of Features Survived: 3
Elite Selected Features: ['bmi', 's1', 's5']

Calculated Cutoff Score was: 354.7051

Importance Breakdown:
  Feature  Absolute Weight (Importance)
8      s5                    669.922675
4      s1                    569.438134
2     bmi                    521.744369
3      bp                    321.060777
5      s2                    302.453193
1     sex                    235.993080
7      s4                    143.698515
9      s6                     66.835511
0     age                      6.494693
6      s3                      0.000000
